# Thermal dataset builder v4

- пошук **цілої області**, а не лише найяскравішої точки
- **CLAHE + DoG** для кращого пошуку на дальній дистанції
- **high/low threshold + region growing**
- **кандидати зі скорингом** за яскравістю, площею, формою, рухом і локальним контрастом
- **motion prediction** за історією центрів
- **dynamic ROI**
- **confirmed / tentative track**
- **far / mid / near** логіка для різних дистанцій
- більш інформативне **preview**


In [ ]:
# 1) Imports + paths
from pathlib import Path
from dataclasses import dataclass, field
from collections import deque
import math
import cv2
import numpy as np

VIDEO_ROOT = Path(r"E:\Drone\VideoFromDrone\Digital")
HUD_VIDEO_DIR = VIDEO_ROOT / "hud"
CLEAN_VIDEO_DIR = VIDEO_ROOT / "clean"

OUT_DIR = Path("dataset_out_v6")
IMAGES_DIR = OUT_DIR / "images"
LABELS_DIR = OUT_DIR / "labels"
PREVIEWS_DIR = OUT_DIR / "previews"
VIDEO_EXTENSIONS = {".avi", ".mp4", ".mov", ".mkv"}

In [13]:

SAVE_NEGATIVE_FRAMES = True
NEGATIVE_SAVE_EVERY = 12
MAX_NEGATIVE_PER_VIDEO = 100

MAKE_PREVIEW = True
PREVIEW_FPS = 10
PREVIEW_EXT = ".mp4"
PREVIEW_CODEC = "mp4v"

DRAW_HUD_RECTS = True
DRAW_ROI = True
DRAW_DEBUG_TEXT = True
DRAW_REJECTED_CANDIDATES = False

CLASS_ID = 0
# Для датасету краще зберігати тільки стабільні треки.
# False/0.18 давали багато хибних bbox на хмарах/березі.
REQUIRE_CONFIRMED_FOR_SAVE = True
MIN_CONF_TO_SAVE = 0.45

# Захист від поганих YOLO-label:
# надто малий bbox, bbox на самому краї або дуже витягнутий bbox не зберігаємо.
MIN_SAVE_BOX_PX = 8
MIN_SAVE_AREA_PX = 16
SAVE_EDGE_MARGIN_PX = 6
MAX_SAVE_ASPECT = 4.0

PROFILE_DEFAULT = "clean"

USE_CLAHE = True
CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)
DOG_SIGMA_SMALL = 1.0
DOG_SIGMA_BIG = 3.0
DOG_WEIGHT = 0.45
ENHANCE_WEIGHT = 0.55
RAW_WEIGHT = 0.45

BASE_BRIGHT_FLOOR = 95
DELTA_FROM_LOCAL_MAX = 28
LOW_THR_OFFSET = 38
MIN_LOW_THR = 55
GAUSS_BLUR = 3

OPEN_KERNEL = 1
CLOSE_KERNEL = 3
DILATE_KERNEL = 3
REGION_GROW_EXPAND = 30

MIN_COMPONENT_PIXELS = 1
EDGE_MARGIN_RATIO = 0.03

FAR_AREA_RANGE = (1, 40)
MID_AREA_RANGE = (4, 120)
NEAR_AREA_RANGE = (12, 500)
MAX_GLOBAL_AREA = 1800

TRACK_HISTORY = 6
TRACK_CONFIRM_FRAMES = 5    # Fix: було 2 — хмара підтверджувалась за 2 кадри
TRACK_KEEP_FRAMES = 6
MAX_MISSED_FRAMES = 8       # Fix: було 12 — швидше скидати хибний трек
BASE_ROI_HALF_SIZE = 110
ROI_GROW_PER_MISS = 36
ROI_MIN_HALF_SIZE = 80
ROI_MAX_HALF_SIZE = 320
MAX_JUMP_PX = 180
MAX_AREA_GROWTH = 3.5       # Fix: було 7.0 — хмари розширюються швидко
MAX_AREA_SHRINK = 0.10

W_MEAN = 0.15               # Fix: знижено з 0.22 — яскравість не відрізняє хмару від цілі
W_PEAK = 0.15
W_CONTRAST = 0.19
W_AREA = 0.10
W_COMPACT = 0.04
W_SOLIDITY = 0.03
W_PROX = 0.18               # Fix: підвищено з 0.15
W_MOTION = 0.20             # Fix: підвищено з 0.10 — хмари рухаються лінійно, ціль — хаотично

# HUD profile
USE_HUD_MASK = True
HUD_TOP_PX = 80
HUD_BOTTOM_PX = 100
HUD_LEFT_PX = 90
HUD_RIGHT_PX = 90

HUD_EXTRA_BLOCKS = [
    # (x1, y1, x2, y2),
]

HUD_EDGE_MARGIN = 40
HUD_MAX_ASPECT = 3.8
HUD_MIN_FILL_RATIO = 0.22
HUD_GLOBAL_SEARCH_AFTER_MISSES = 6

HUD_W_MEAN = 0.14
HUD_W_PEAK = 0.08
HUD_W_CONTRAST = 0.18
HUD_W_AREA = 0.10
HUD_W_COMPACT = 0.06
HUD_W_SOLIDITY = 0.05
HUD_W_PROX = 0.24
HUD_W_MOTION = 0.15

HUD_MAX_JUMP_PX = 90
HUD_TRACK_CONFIRM_FRAMES = 5    # Fix: було 3

HUD_OPEN_KERNEL = 3
HUD_CLOSE_KERNEL = 5
HUD_DILATE_KERNEL = 3

HUD_BASE_BRIGHT_FLOOR = 105
HUD_MIN_LOW_THR = 60


In [ ]:
# 3) Data structures + helpers
@dataclass
class BBox:
    x1: int
    y1: int
    x2: int
    y2: int

    @property
    def w(self):
        return max(0, self.x2 - self.x1)

    @property
    def h(self):
        return max(0, self.y2 - self.y1)

    @property
    def area(self):
        return self.w * self.h

    @property
    def cx(self):
        return 0.5 * (self.x1 + self.x2)

    @property
    def cy(self):
        return 0.5 * (self.y1 + self.y2)

    def as_int(self):
        return (int(self.x1), int(self.y1), int(self.x2), int(self.y2))


@dataclass
class Candidate:
    bbox: BBox
    score: float
    mean_intensity: float
    peak_intensity: float
    local_contrast: float
    compactness: float
    solidity: float
    aspect_ratio: float
    seed_area: int
    grown_area: int
    mode: str
    centroid: tuple[float, float]
    debug: dict = field(default_factory=dict)


@dataclass
class TrackState:
    bbox: BBox | None = None
    score: float = 0.0
    confirmed: bool = False
    confirm_hits: int = 0
    weak_hits: int = 0
    missed: int = 0
    history: deque = field(default_factory=lambda: deque(maxlen=TRACK_HISTORY))
    area_history: deque = field(default_factory=lambda: deque(maxlen=TRACK_HISTORY))
    last_mode: str = "global"
    last_roi: tuple[int, int, int, int] | None = None

    def push(self, bbox: BBox, score: float):
        self.bbox = bbox
        self.score = float(score)
        self.history.append((bbox.cx, bbox.cy))
        self.area_history.append(max(1.0, float(bbox.area)))


def ensure_dirs():
    for p in [OUT_DIR, IMAGES_DIR, LABELS_DIR, PREVIEWS_DIR]:
        p.mkdir(parents=True, exist_ok=True)


def collect_video_jobs():
    jobs = []

    if HUD_VIDEO_DIR.exists():
        for p in sorted(HUD_VIDEO_DIR.rglob("*")):
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS:
                jobs.append((p, "hud", "auto_all"))

    if CLEAN_VIDEO_DIR.exists():
        for p in sorted(CLEAN_VIDEO_DIR.rglob("*")):
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS:
                jobs.append((p, "clean", "none"))

    return jobs


def clip_int(v, lo, hi):
    return int(max(lo, min(hi, v)))


def clamp01(x):
    return float(max(0.0, min(1.0, x)))


def safe_div(a, b, default=0.0):
    return a / b if abs(b) > 1e-9 else default


def expand_bbox(bbox: BBox, pad: int, frame_w: int, frame_h: int) -> BBox:
    return BBox(
        clip_int(bbox.x1 - pad, 0, frame_w - 1),
        clip_int(bbox.y1 - pad, 0, frame_h - 1),
        clip_int(bbox.x2 + pad, 1, frame_w),
        clip_int(bbox.y2 + pad, 1, frame_h),
    )


def bbox_from_mask(mask, offset_x=0, offset_y=0):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    return BBox(x1 + offset_x, y1 + offset_y, x2 + offset_x, y2 + offset_y)


def yolo_line_from_bbox(bbox, frame_w, frame_h, class_id=0):
    xc = ((bbox.x1 + bbox.x2) / 2) / frame_w
    yc = ((bbox.y1 + bbox.y2) / 2) / frame_h
    bw = (bbox.x2 - bbox.x1) / frame_w
    bh = (bbox.y2 - bbox.y1) / frame_h
    return f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"

def prepare_bbox_for_yolo(bbox: BBox, frame_w: int, frame_h: int):

    if bbox is None:
        return None

    x1, y1, x2, y2 = bbox.as_int()
    w = max(0, x2 - x1)
    h = max(0, y2 - y1)

    if w <= 0 or h <= 0:
        return None

    if x1 <= SAVE_EDGE_MARGIN_PX or y1 <= SAVE_EDGE_MARGIN_PX:
        return None
    if x2 >= frame_w - SAVE_EDGE_MARGIN_PX or y2 >= frame_h - SAVE_EDGE_MARGIN_PX:
        return None

    aspect = max(w / max(1, h), h / max(1, w))
    if aspect > MAX_SAVE_ASPECT:
        return None

    if w * h < MIN_SAVE_AREA_PX:
        return None

    # Якщо об'єкт дуже малий, bbox робимо мінімально придатним для YOLO.
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    new_w = max(w, MIN_SAVE_BOX_PX)
    new_h = max(h, MIN_SAVE_BOX_PX)

    nx1 = clip_int(round(cx - new_w / 2), 0, frame_w - 1)
    ny1 = clip_int(round(cy - new_h / 2), 0, frame_h - 1)
    nx2 = clip_int(round(cx + new_w / 2), 1, frame_w)
    ny2 = clip_int(round(cy + new_h / 2), 1, frame_h)

    if nx2 <= nx1 or ny2 <= ny1:
        return None

    return BBox(nx1, ny1, nx2, ny2)


In [15]:

# 4) HUD masking + preprocessing
def get_hud_rects(frame, hud_mode="auto_all"):
    h, w = frame.shape[:2]
    if hud_mode == "none":
        return []

    rects = []
    rects.append((0, 0, w, int(h * 0.20)))
    rects.append((0, 0, int(w * 0.22), int(h * 0.58)))
    rects.append((int(w * 0.78), 0, w, int(h * 0.58)))
    rects.append((int(w * 0.41), int(h * 0.36), int(w * 0.59), int(h * 0.69)))
    rects.append((int(w * 0.12), int(h * 0.71), int(w * 0.88), h))
    return rects


def build_hud_mask(shape, auto_rects=None):
    h, w = shape[:2]
    mask = np.ones((h, w), dtype=np.uint8) * 255

    if not USE_HUD_MASK:
        return mask

    if HUD_TOP_PX > 0:
        mask[:HUD_TOP_PX, :] = 0
    if HUD_BOTTOM_PX > 0:
        mask[h - HUD_BOTTOM_PX:, :] = 0
    if HUD_LEFT_PX > 0:
        mask[:, :HUD_LEFT_PX] = 0
    if HUD_RIGHT_PX > 0:
        mask[:, w - HUD_RIGHT_PX:] = 0

    if auto_rects:
        for x1, y1, x2, y2 in auto_rects:
            mask[y1:y2, x1:x2] = 0

    for x1, y1, x2, y2 in HUD_EXTRA_BLOCKS:
        x1 = max(0, min(w, x1))
        x2 = max(0, min(w, x2))
        y1 = max(0, min(h, y1))
        y2 = max(0, min(h, y2))
        mask[y1:y2, x1:x2] = 0

    return mask


def apply_hud_mask(gray, mask):
    return cv2.bitwise_and(gray, gray, mask=mask)


def preprocess_frame(frame, hud_rects, profile="clean"):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    hud_mask = np.ones_like(gray, dtype=np.uint8) * 255
    if profile == "hud":
        hud_mask = build_hud_mask(gray.shape, hud_rects)
        gray = apply_hud_mask(gray, hud_mask)

    if GAUSS_BLUR > 1:
        gray = cv2.GaussianBlur(gray, (GAUSS_BLUR, GAUSS_BLUR), 0)

    raw = gray.astype(np.uint8)

    if USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
        clahe_img = clahe.apply(raw)
    else:
        clahe_img = raw

    dog_small = cv2.GaussianBlur(clahe_img, (0, 0), DOG_SIGMA_SMALL)
    dog_big = cv2.GaussianBlur(clahe_img, (0, 0), DOG_SIGMA_BIG)
    dog = cv2.subtract(dog_small, dog_big)

    enhanced = cv2.addWeighted(clahe_img, ENHANCE_WEIGHT, dog, DOG_WEIGHT, 0)
    fused = cv2.addWeighted(raw, RAW_WEIGHT, enhanced, 1.0 - RAW_WEIGHT, 0)

    return {
        "gray": raw,
        "clahe": clahe_img,
        "dog": dog,
        "enhanced": enhanced,
        "fused": fused,
        "profile": profile,
        "hud_mask": hud_mask,
        "hud_rects": hud_rects,
    }


In [16]:
# 5) ROI, motion prediction, distance modes
def predict_center(track: TrackState):
    if len(track.history) == 0:
        return None
    if len(track.history) == 1:
        return track.history[-1]

    pts = list(track.history)
    vx = np.mean([pts[i][0] - pts[i - 1][0] for i in range(1, len(pts))])
    vy = np.mean([pts[i][1] - pts[i - 1][1] for i in range(1, len(pts))])
    px = pts[-1][0] + vx
    py = pts[-1][1] + vy
    return (px, py)


def estimate_distance_mode(track: TrackState):
    if track.bbox is None:
        return "far"
    a = max(1, track.bbox.area)
    if a <= FAR_AREA_RANGE[1]:
        return "far"
    if a <= MID_AREA_RANGE[1]:
        return "mid"
    return "near"


def get_area_range(mode: str):
    if mode == "far":
        return FAR_AREA_RANGE
    if mode == "mid":
        return MID_AREA_RANGE
    return NEAR_AREA_RANGE


def build_search_roi(frame_w, frame_h, track: TrackState):
    pred = predict_center(track)
    if pred is None or track.bbox is None:
        return None

    base = max(track.bbox.w, track.bbox.h, BASE_ROI_HALF_SIZE)
    half = int(base * 0.8 + ROI_GROW_PER_MISS * track.missed)
    half = clip_int(half, ROI_MIN_HALF_SIZE, ROI_MAX_HALF_SIZE)

    cx, cy = pred
    x1 = clip_int(cx - half, 0, frame_w - 1)
    y1 = clip_int(cy - half, 0, frame_h - 1)
    x2 = clip_int(cx + half, 1, frame_w)
    y2 = clip_int(cy + half, 1, frame_h)

    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return (x1, y1, x2, y2)

In [ ]:

# 6) Candidate extraction: thresholds, whole-object growing, scoring
def bbox_near_edge(x, y, w, h, W, H, margin):
    return (
        x < margin or
        y < margin or
        x + w > W - margin or
        y + h > H - margin
    )


def candidate_fill_ratio(area, w, h):
    return area / max(1.0, w * h)


def should_reject_hud_candidate(x, y, w, h, area, W, H):
    if w <= 0 or h <= 0:
        return True

    aspect = max(w / max(1.0, h), h / max(1.0, w))
    fill_ratio = candidate_fill_ratio(area, w, h)

    if bbox_near_edge(x, y, w, h, W, H, HUD_EDGE_MARGIN):
        return True

    if aspect > HUD_MAX_ASPECT:
        return True

    if fill_ratio < HUD_MIN_FILL_RATIO:
        return True

    return False


def make_binary_masks(proc, roi_bbox=None, distance_mode="far"):
    fused = proc["fused"]
    gray = proc["gray"]
    profile = proc.get("profile", PROFILE_DEFAULT)

    if roi_bbox is None:
        x1, y1, x2, y2 = 0, 0, fused.shape[1], fused.shape[0]
    else:
        x1, y1, x2, y2 = roi_bbox

    roi_fused = fused[y1:y2, x1:x2]
    roi_gray = gray[y1:y2, x1:x2]
    if roi_fused.size == 0:
        return None

    local_max = int(roi_fused.max())
    local_mean = float(roi_fused.mean())
    local_std = float(roi_fused.std())

    bright_floor = HUD_BASE_BRIGHT_FLOOR if profile == "hud" else BASE_BRIGHT_FLOOR
    min_low_thr = HUD_MIN_LOW_THR if profile == "hud" else MIN_LOW_THR

    hi_thr = max(bright_floor, local_max - DELTA_FROM_LOCAL_MAX)
    if distance_mode == "far":  # Fix: на far-дистанції беремо тільки найяскравіші точки
        hi_thr = max(hi_thr, int(local_mean + 2.0 * local_std))
    else:
        hi_thr = max(hi_thr, int(local_mean + 1.2 * local_std))
    lo_thr = max(min_low_thr, hi_thr - LOW_THR_OFFSET)

    _, seed = cv2.threshold(roi_fused, hi_thr, 255, cv2.THRESH_BINARY)
    _, low = cv2.threshold(roi_fused, lo_thr, 255, cv2.THRESH_BINARY)

    open_kernel = HUD_OPEN_KERNEL if profile == "hud" else OPEN_KERNEL
    close_kernel = HUD_CLOSE_KERNEL if profile == "hud" else CLOSE_KERNEL

    if open_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel, open_kernel))
        seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN, k)
    if close_kernel > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
        low = cv2.morphologyEx(low, cv2.MORPH_CLOSE, k)

    return {
        "roi_bbox": (x1, y1, x2, y2),
        "seed_mask": seed,
        "low_mask": low,
        "roi_fused": roi_fused,
        "roi_gray": roi_gray,
        "hi_thr": hi_thr,
        "lo_thr": lo_thr,
        "local_mean": local_mean,
        "local_std": local_std,
        "distance_mode": distance_mode,
    }


def region_grow_from_seed(seed_mask, low_mask, contour_mask, profile="clean"):
    allowed = cv2.bitwise_and(low_mask, contour_mask)
    seed_inside = cv2.bitwise_and(seed_mask, contour_mask)
    if seed_inside.max() == 0:
        return None

    n, labels, stats, _ = cv2.connectedComponentsWithStats(allowed, connectivity=8)
    seed_labels = np.unique(labels[seed_inside > 0])
    seed_labels = [lab for lab in seed_labels if lab != 0]
    if not seed_labels:
        return None

    grown = np.zeros_like(allowed)
    for lab in seed_labels:
        grown[labels == lab] = 255

    dilate_kernel = HUD_DILATE_KERNEL if profile == "hud" else DILATE_KERNEL
    if dilate_kernel > 1:
        k = np.ones((dilate_kernel, dilate_kernel), np.uint8)
        grown = cv2.dilate(grown, k, iterations=1)
        grown = cv2.bitwise_and(grown, allowed)

    return grown


def contour_features(contour, grown_mask, roi_gray, roi_fused):
    x, y, w, h = cv2.boundingRect(contour)
    bbox = BBox(x, y, x + w, y + h)

    contour_area = max(1.0, cv2.contourArea(contour))
    perimeter = max(1.0, cv2.arcLength(contour, True))
    hull = cv2.convexHull(contour)
    hull_area = max(1.0, cv2.contourArea(hull))
    compactness = float(4.0 * math.pi * contour_area / (perimeter * perimeter))
    solidity = float(contour_area / hull_area)
    aspect = float(w / max(1, h))

    crop_gray = roi_gray[y:y+h, x:x+w]
    crop_fused = roi_fused[y:y+h, x:x+w]
    crop_mask = grown_mask[y:y+h, x:x+w] > 0
    if crop_mask.sum() == 0:
        return None

    vals_gray = crop_gray[crop_mask]
    vals_fused = crop_fused[crop_mask]
    mean_intensity = float(vals_gray.mean())
    peak_intensity = float(vals_gray.max())
    local_bg = float(np.median(crop_gray))
    local_contrast = float(vals_gray.mean() - local_bg)
    grown_area = int(crop_mask.sum())

    ys, xs = np.where(crop_mask)
    cx = x + float(xs.mean())
    cy = y + float(ys.mean())

    return {
        "bbox": bbox,
        "mean_intensity": mean_intensity,
        "peak_intensity": peak_intensity,
        "local_contrast": local_contrast,
        "compactness": compactness,
        "solidity": solidity,
        "aspect_ratio": aspect,
        "seed_area": int(contour_area),
        "grown_area": grown_area,
        "centroid": (cx, cy),
    }


def score_candidate(features, track: TrackState, roi_offset=(0, 0), frame_shape=None, profile="clean"):
    bbox = features["bbox"]
    mean_n = clamp01((features["mean_intensity"] - 90) / 120)
    peak_n = clamp01((features["peak_intensity"] - 120) / 110)
    contrast_n = clamp01((features["local_contrast"] + 10) / 55)

    mode = estimate_distance_mode(track)
    a_min, a_max = get_area_range(mode)
    area = features["grown_area"]
    if area < a_min:
        area_score = clamp01(area / max(1, a_min))
    elif area > a_max:
        area_score = clamp01(1.0 - (area - a_max) / max(1, (a_max * 2)))
    else:
        area_score = 1.0

    compact_n = clamp01(features["compactness"] / 0.85)
    solidity_n = clamp01(features["solidity"])
    aspect_penalty = abs(math.log(max(1e-6, features["aspect_ratio"])))
    shape_bonus = clamp01(1.0 - aspect_penalty / 1.5)

    prox = 0.5
    motion = 0.5
    max_jump_px = HUD_MAX_JUMP_PX if profile == "hud" else MAX_JUMP_PX

    if track.bbox is not None:
        dist = math.hypot(bbox.cx - track.bbox.cx, bbox.cy - track.bbox.cy)
        prox = clamp01(1.0 - dist / max_jump_px)

    pred = predict_center(track)
    if pred is not None:
        md = math.hypot(bbox.cx - pred[0], bbox.cy - pred[1])
        motion = clamp01(1.0 - md / max_jump_px)

    if profile == "hud":
        w_mean = HUD_W_MEAN
        w_peak = HUD_W_PEAK
        w_contrast = HUD_W_CONTRAST
        w_area = HUD_W_AREA
        w_compact = HUD_W_COMPACT
        w_solidity = HUD_W_SOLIDITY
        w_prox = HUD_W_PROX
        w_motion = HUD_W_MOTION
    else:
        w_mean = W_MEAN
        w_peak = W_PEAK
        w_contrast = W_CONTRAST
        w_area = W_AREA
        w_compact = W_COMPACT
        w_solidity = W_SOLIDITY
        w_prox = W_PROX
        w_motion = W_MOTION

    edge_penalty = 0.0
    if frame_shape is not None:
        H, W = frame_shape[:2]
        mx = int(W * EDGE_MARGIN_RATIO)
        my = int(H * EDGE_MARGIN_RATIO)
        if bbox.x1 <= mx or bbox.y1 <= my or bbox.x2 >= W - mx or bbox.y2 >= H - my:
            edge_penalty = 0.12 if profile != "hud" else 0.0

    # Fix: штраф за великі рівномірні регіони (хмари — великі, солідні, рівномірні)
    cloud_penalty = 0.0
    if features["grown_area"] > 200:
        cloud_penalty = 0.20
    elif features["grown_area"] > 80 and features["solidity"] > 0.85:
        cloud_penalty = 0.15

    score = (
        w_mean * mean_n
        + w_peak * peak_n
        + w_contrast * contrast_n
        + w_area * area_score
        + w_compact * compact_n
        + w_solidity * ((solidity_n + shape_bonus) * 0.5)
        + w_prox * prox
        + w_motion * motion
        - edge_penalty
        - cloud_penalty  # Fix: штраф за хмари/великі рівномірні зони
    )

    score = float(max(0.0, min(1.0, score)))
    return score


def extract_candidates(proc, track: TrackState, roi_bbox=None, search_mode="global"):
    profile = proc.get("profile", PROFILE_DEFAULT)
    masks = make_binary_masks(proc, roi_bbox=roi_bbox, distance_mode=estimate_distance_mode(track))
    if masks is None:
        return []

    seed_mask = masks["seed_mask"]
    low_mask = masks["low_mask"]
    roi_gray = masks["roi_gray"]
    roi_fused = masks["roi_fused"]
    xoff, yoff, _, _ = masks["roi_bbox"]

    cnts, _ = cv2.findContours(seed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []

    for c in cnts:
        seed_area = cv2.contourArea(c)
        if seed_area < MIN_COMPONENT_PIXELS:
            continue
        if seed_area > MAX_GLOBAL_AREA:
            continue

        contour_mask = np.zeros_like(seed_mask)
        cv2.drawContours(contour_mask, [c], -1, 255, thickness=-1)

        grown = region_grow_from_seed(seed_mask, low_mask, contour_mask, profile=profile)
        if grown is None:
            continue

        grown_bbox_local = bbox_from_mask(grown)
        if grown_bbox_local is None:
            continue

        grow_pad = expand_bbox(grown_bbox_local, REGION_GROW_EXPAND, roi_fused.shape[1], roi_fused.shape[0])
        sub_seed = seed_mask[grow_pad.y1:grow_pad.y2, grow_pad.x1:grow_pad.x2]
        sub_low = low_mask[grow_pad.y1:grow_pad.y2, grow_pad.x1:grow_pad.x2]
        if sub_seed.size == 0:
            continue

        allowed = sub_low.copy()
        cc = region_grow_from_seed(sub_seed, allowed, np.full_like(sub_seed, 255), profile=profile)
        grown_refined_local = np.zeros_like(seed_mask)
        if cc is not None:
            grown_refined_local[grow_pad.y1:grow_pad.y2, grow_pad.x1:grow_pad.x2] = cc
        else:
            grown_refined_local = grown

        grown_contours, _ = cv2.findContours(grown_refined_local, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not grown_contours:
            continue
        main_contour = max(grown_contours, key=cv2.contourArea)

        feats = contour_features(main_contour, grown_refined_local, roi_gray, roi_fused)
        if feats is None:
            continue

        global_bbox = BBox(
            feats["bbox"].x1 + xoff,
            feats["bbox"].y1 + yoff,
            feats["bbox"].x2 + xoff,
            feats["bbox"].y2 + yoff,
        )
        feats["bbox"] = global_bbox
        feats["centroid"] = (feats["centroid"][0] + xoff, feats["centroid"][1] + yoff)

        if profile == "hud":
            if should_reject_hud_candidate(
                global_bbox.x1,
                global_bbox.y1,
                global_bbox.w,
                global_bbox.h,
                feats["grown_area"],
                proc["gray"].shape[1],
                proc["gray"].shape[0],
            ):
                continue

        score = score_candidate(feats, track, frame_shape=proc["gray"].shape, profile=profile)
        mode = estimate_distance_mode(track)
        a_min, a_max = get_area_range(mode)
        if feats["grown_area"] < max(1, a_min // 2):
            continue
        if feats["grown_area"] > a_max * 3:
            continue

        cand = Candidate(
            bbox=global_bbox,
            score=score,
            mean_intensity=feats["mean_intensity"],
            peak_intensity=feats["peak_intensity"],
            local_contrast=feats["local_contrast"],
            compactness=feats["compactness"],
            solidity=feats["solidity"],
            aspect_ratio=feats["aspect_ratio"],
            seed_area=feats["seed_area"],
            grown_area=feats["grown_area"],
            mode=search_mode,
            centroid=feats["centroid"],
            debug={
                "hi_thr": masks["hi_thr"],
                "lo_thr": masks["lo_thr"],
                "distance_mode": mode,
                "profile": profile,
            }
        )
        candidates.append(cand)

    candidates.sort(key=lambda c: c.score, reverse=True)
    return candidates


def validate_candidate(cand: Candidate, track: TrackState, profile="clean"):
    max_jump_px = HUD_MAX_JUMP_PX if profile == "hud" else MAX_JUMP_PX

    if track.bbox is None:
        return True

    jump = math.hypot(cand.bbox.cx - track.bbox.cx, cand.bbox.cy - track.bbox.cy)
    if jump > max_jump_px * (1.0 + 0.25 * track.missed):
        return False

    # Fix: перевірка відхилення від передбаченої позиції (хмари рухаються лінійно)
    pred = predict_center(track)
    if pred is not None and len(track.history) >= 3:
        pred_dist = math.hypot(cand.bbox.cx - pred[0], cand.bbox.cy - pred[1])
        if pred_dist > max_jump_px * 0.6:
            return False

    prev_area = max(1.0, track.bbox.area)
    ratio = cand.bbox.area / prev_area
    if ratio > MAX_AREA_GROWTH or ratio < MAX_AREA_SHRINK:
        return False

    return True


def choose_best_candidate(proc, track: TrackState):
    H, W = proc["gray"].shape[:2]
    profile = proc.get("profile", PROFILE_DEFAULT)
    roi = build_search_roi(W, H, track)
    candidates = []

    if roi is not None:
        candidates.extend(extract_candidates(proc, track, roi_bbox=roi, search_mode="roi"))

    use_global = False
    if track.bbox is None:
        use_global = True
    elif profile == "hud":
        use_global = track.missed >= HUD_GLOBAL_SEARCH_AFTER_MISSES
    else:
        use_global = (not candidates) or (candidates and candidates[0].score < 0.28) or track.missed >= 2

    if use_global:
        extra = extract_candidates(proc, track, roi_bbox=None, search_mode="global")
        candidates.extend(extra)

    dedup = []
    for cand in sorted(candidates, key=lambda c: c.score, reverse=True):
        keep = True
        for ex in dedup:
            dist = math.hypot(cand.bbox.cx - ex.bbox.cx, cand.bbox.cy - ex.bbox.cy)
            if dist < max(8, 0.4 * max(cand.bbox.w, cand.bbox.h, ex.bbox.w, ex.bbox.h)):
                keep = False
                break
        if keep:
            dedup.append(cand)

    for cand in dedup:
        if validate_candidate(cand, track, profile=profile):
            return cand, dedup, roi

    return None, dedup, roi


In [18]:

# 7) Track management + preview
def update_track(track: TrackState, cand: Candidate | None, profile="clean"):
    confirm_frames = HUD_TRACK_CONFIRM_FRAMES if profile == "hud" else TRACK_CONFIRM_FRAMES

    if cand is None:
        track.missed += 1
        track.weak_hits = 0
        if track.missed > TRACK_KEEP_FRAMES:
            track.confirm_hits = max(0, track.confirm_hits - 1)
        if track.missed > MAX_MISSED_FRAMES:
            track.confirmed = False
            track.bbox = None
        return track

    track.push(cand.bbox, cand.score)
    track.last_mode = cand.mode
    track.missed = 0

    if cand.score >= 0.45:
        track.confirm_hits += 1
        track.weak_hits = 0
    elif cand.score >= 0.28:
        track.weak_hits += 1
        track.confirm_hits = max(1, track.confirm_hits)
    else:
        track.weak_hits += 1

    if track.confirm_hits >= confirm_frames:
        track.confirmed = True

    return track


def should_save_positive(track: TrackState, cand: Candidate | None, frame_w=None, frame_h=None):
    if cand is None:
        return False
    if cand.score < MIN_CONF_TO_SAVE:
        return False
    if REQUIRE_CONFIRMED_FOR_SAVE and not track.confirmed:
        return False

    # Не зберігаємо label, якщо bbox непридатний для YOLO.
    if frame_w is not None and frame_h is not None:
        safe_bbox = prepare_bbox_for_yolo(cand.bbox, frame_w, frame_h)
        if safe_bbox is None:
            return False

    return True


def make_preview_path(video_path, source_tag):
    return PREVIEWS_DIR / f"{source_tag}_{video_path.stem}_preview{PREVIEW_EXT}"


def create_preview_writer(preview_path, frame_w, frame_h, fps=10):
    fourcc = cv2.VideoWriter_fourcc(*PREVIEW_CODEC)
    writer = cv2.VideoWriter(str(preview_path), fourcc, fps, (frame_w, frame_h))
    if not writer.isOpened():
        raise RuntimeError(f"Preview writer was not opened: {preview_path}")
    return writer


def draw_preview_frame(frame, track: TrackState, cand: Candidate | None, all_candidates=None, hud_rects=None, roi=None, frame_idx=None, source_tag=None, profile="clean"):
    vis = frame.copy()

    if DRAW_HUD_RECTS and profile == "hud" and hud_rects:
        for x1, y1, x2, y2 in hud_rects:
            cv2.rectangle(vis, (x1, y1), (x2, y2), (70, 70, 70), 1)

    if DRAW_ROI and roi is not None:
        x1, y1, x2, y2 = roi
        cv2.rectangle(vis, (x1, y1), (x2, y2), (120, 120, 120), 1)

    if all_candidates:
        for i, cc in enumerate(all_candidates[:5]):
            b = cc.bbox
            cv2.rectangle(vis, (b.x1, b.y1), (b.x2, b.y2), (120, 120, 120), 1)

    pred = predict_center(track)
    if pred is not None:
        px, py = int(pred[0]), int(pred[1])
        cv2.drawMarker(vis, (px, py), (180, 180, 180), markerType=cv2.MARKER_CROSS, markerSize=12, thickness=1)

    if cand is not None:
        b = cand.bbox
        thickness = 3 if track.confirmed else 2
        cv2.rectangle(vis, (b.x1, b.y1), (b.x2, b.y2), (255, 255, 255), thickness)
        label = f"{cand.mode} s={cand.score:.2f}"
        cv2.putText(vis, label, (max(8, b.x1), max(18, b.y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
    else:
        cv2.putText(vis, "no target", (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (180, 180, 180), 2, cv2.LINE_AA)

    if DRAW_DEBUG_TEXT:
        lines = [
            f"src={source_tag}" if source_tag is not None else None,
            f"profile={profile}",
            f"frame={frame_idx}" if frame_idx is not None else None,
            f"confirmed={track.confirmed}",
            f"hits={track.confirm_hits}",
            f"missed={track.missed}",
            f"dist_mode={estimate_distance_mode(track)}",
        ]
        lines = [x for x in lines if x is not None]
        y = vis.shape[0] - 12 - 18 * (len(lines) - 1)
        for line in lines:
            cv2.putText(vis, line, (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (210, 210, 210), 2, cv2.LINE_AA)
            y += 18

    return vis


In [19]:
# 8) Main processing
ensure_dirs()
summary = []

jobs = collect_video_jobs()
print("Found videos:", len(jobs))

for video_path, source_tag, forced_mode in jobs:
    print(f"\nProcessing: {video_path.name} | source={source_tag}")

    cap = cv2.VideoCapture(str(video_path))
    ok, first = cap.read()
    if not ok:
        print("  skip: cannot read video")
        cap.release()
        continue

    frame_h, frame_w = first.shape[:2]
    hud_rects = get_hud_rects(first, forced_mode)

    preview_path = make_preview_path(video_path, source_tag)
    preview_writer = None
    if MAKE_PREVIEW:
        preview_writer = create_preview_writer(preview_path, frame_w, frame_h, PREVIEW_FPS)

    track = TrackState()
    idx = 0
    saved_pos = 0
    saved_neg = 0

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        profile = "hud" if source_tag == "hud" else "clean"
        proc = preprocess_frame(frame, hud_rects, profile=profile)
        cand, all_candidates, roi = choose_best_candidate(proc, track)
        track.last_roi = roi
        track = update_track(track, cand, profile=profile)

        if should_save_positive(track, cand, frame_w, frame_h):
            safe_bbox = prepare_bbox_for_yolo(cand.bbox, frame_w, frame_h)
            if safe_bbox is not None:
                stem = f"{source_tag}_{video_path.stem}_{idx:06d}"
                img = IMAGES_DIR / f"{stem}.jpg"
                lbl = LABELS_DIR / f"{stem}.txt"
                cv2.imwrite(str(img), frame)
                with open(lbl, "w", encoding="utf-8") as f:
                    f.write(yolo_line_from_bbox(safe_bbox, frame_w, frame_h, CLASS_ID) + "\n")
                saved_pos += 1

        elif SAVE_NEGATIVE_FRAMES and saved_neg < MAX_NEGATIVE_PER_VIDEO and idx % NEGATIVE_SAVE_EVERY == 0:
            stem = f"{source_tag}_{video_path.stem}_{idx:06d}_neg"
            img = IMAGES_DIR / f"{stem}.jpg"
            lbl = LABELS_DIR / f"{stem}.txt"
            cv2.imwrite(str(img), frame)
            with open(lbl, "w", encoding="utf-8") as f:
                f.write("")
            saved_neg += 1

        if preview_writer is not None:
            vis = draw_preview_frame(
                frame,
                track=track,
                cand=cand,
                all_candidates=all_candidates,
                hud_rects=hud_rects,
                roi=roi,
                frame_idx=idx,
                source_tag=source_tag,
                profile=profile,
            )
            preview_writer.write(vis)

        if idx % 150 == 0:
            top_score = cand.score if cand is not None else 0.0
            print(f"  frame={idx:6d} | confirmed={track.confirmed} | score={top_score:.2f} | saved={saved_pos}")

        idx += 1

    cap.release()
    if preview_writer is not None:
        preview_writer.release()

    summary.append({
        "video": video_path.name,
        "source": source_tag,
        "frames": idx,
        "saved_positive": saved_pos,
        "saved_negative": saved_neg,
        "preview": str(preview_path) if MAKE_PREVIEW else None,
    })

summary

Found videos: 0


[]

In [20]:
# 9) Quick summary
print("Processed videos:", len(summary))
for row in summary[:20]:
    print(row)

Processed videos: 0


# 9.5) Dataset split for YOLO

Цей блок:
- бере **100% позитивних** кадрів
- бере **потрібну кількість пустих** кадрів
- ділить дані на **train / val / test**
- копіює файли у правильну структуру папок YOLO
- створює `data.yaml`

Можна працювати у двох режимах для пустих кадрів:
1. `NEGATIVE_MODE = "ratio"` — кількість пустих кадрів як частка від позитивних
2. `NEGATIVE_MODE = "fixed"` — фіксована кількість пустих кадрів

За замовчуванням:
- усі позитивні кадри використовуються
- пусті кадри беруться як `0.5` від позитивних
- split: `80 / 20 / 0`


In [21]:

# 9.6) Split exported dataset into train / val / test
import random
import shutil
import re
from pathlib import Path

RANDOM_SEED = 42

# ---- split sizes ----
TRAIN_RATIO = 0.80
VAL_RATIO = 0.20
TEST_RATIO = 0.00   # можеш поставити 0.10, якщо потрібен test

# ---- negative sampling ----
USE_ALL_POSITIVES = True

NEGATIVE_MODE = "ratio"   # "ratio" або "fixed"
NEGATIVE_RATIO_TO_POS = 0.25   # якщо mode="ratio": 0.5 = 1 пустий на 2 позитивних
NEGATIVE_FIXED_COUNT = 300     # якщо mode="fixed"

# ---- output ----
YOLOSET_DIR = OUT_DIR / "yolo_split"
CREATE_DATA_YAML = True
CLASS_NAMES = ["target"]

def list_exported_pairs(images_dir: Path, labels_dir: Path):
    pairs = []
    for img in sorted(images_dir.glob("*.jpg")):
        lbl = labels_dir / f"{img.stem}.txt"
        if lbl.exists():
            pairs.append((img, lbl))
    return pairs

def is_positive_label(label_path: Path):
    txt = label_path.read_text(encoding="utf-8").strip()
    return len(txt) > 0

def source_video_key(pair):
    """
    Ключ відео для split без витоку кадрів.
    Із stem типу clean_videoName_001234 або clean_videoName_001234_neg
    повертає clean_videoName.
    """
    img, _ = pair
    return re.sub(r"_\d{6}(_neg)?$", "", img.stem)

def split_by_video_groups(items, train_ratio, val_ratio, test_ratio, rng):
    groups = {}
    for pair in items:
        groups.setdefault(source_video_key(pair), []).append(pair)

    group_items = list(groups.items())
    rng.shuffle(group_items)

    n = len(group_items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    if test_ratio == 0:
        n_test = 0
        n_val = n - n_train
    else:
        n_test = n - n_train - n_val

    def flatten(chunk):
        out = []
        for _, pairs in chunk:
            out.extend(pairs)
        return out

    train_items = flatten(group_items[:n_train])
    val_items = flatten(group_items[n_train:n_train + n_val])
    test_items = flatten(group_items[n_train + n_val:n_train + n_val + n_test])
    return train_items, val_items, test_items

def safe_clear_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def prepare_split_dirs(root: Path):
    for split in ["train", "val", "test"]:
        (root / split / "images").mkdir(parents=True, exist_ok=True)
        (root / split / "labels").mkdir(parents=True, exist_ok=True)

def split_list(items, train_ratio, val_ratio, test_ratio, rng):
    items = list(items)
    rng.shuffle(items)

    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    if test_ratio == 0:
        n_test = 0
        n_val = n - n_train

    train_items = items[:n_train]
    val_items = items[n_train:n_train + n_val]
    test_items = items[n_train + n_val:n_train + n_val + n_test]
    return train_items, val_items, test_items

def copy_pairs(pairs, split_name, root: Path):
    for img, lbl in pairs:
        shutil.copy2(img, root / split_name / "images" / img.name)
        shutil.copy2(lbl, root / split_name / "labels" / lbl.name)

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "TRAIN/VAL/TEST ratios must sum to 1.0"

all_pairs = list_exported_pairs(IMAGES_DIR, LABELS_DIR)
positive_pairs = [p for p in all_pairs if is_positive_label(p[1])]
negative_pairs = [p for p in all_pairs if not is_positive_label(p[1])]

print("All exported pairs:", len(all_pairs))
print("Positive pairs:", len(positive_pairs))
print("Negative pairs:", len(negative_pairs))

rng = random.Random(RANDOM_SEED)

# Fix: обидві гілки були однакові — тепер else робить випадкову вибірку
if USE_ALL_POSITIVES:
    selected_positive = list(positive_pairs)
else:
    max_pos = max(1, int(len(positive_pairs) * 0.5))
    selected_positive = rng.sample(positive_pairs, min(max_pos, len(positive_pairs)))

if NEGATIVE_MODE == "ratio":
    target_negative_count = int(len(selected_positive) * NEGATIVE_RATIO_TO_POS)
elif NEGATIVE_MODE == "fixed":
    target_negative_count = int(NEGATIVE_FIXED_COUNT)
else:
    raise ValueError("NEGATIVE_MODE must be 'ratio' or 'fixed'")

target_negative_count = min(target_negative_count, len(negative_pairs))
selected_negative = rng.sample(negative_pairs, target_negative_count) if target_negative_count > 0 else []

print("Selected positives:", len(selected_positive))
print("Selected negatives:", len(selected_negative))

# ВАЖЛИВО: ділимо по відео, а не випадково по кадрах.
# Інакше майже однакові кадри з одного відео потрапляють і в train, і в val.
train_pos, val_pos, test_pos = split_by_video_groups(selected_positive, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, rng)
train_neg, val_neg, test_neg = split_by_video_groups(selected_negative, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, rng)

final_train = train_pos + train_neg
final_val = val_pos + val_neg
final_test = test_pos + test_neg

rng.shuffle(final_train)
rng.shuffle(final_val)
rng.shuffle(final_test)

safe_clear_dir(YOLOSET_DIR)
prepare_split_dirs(YOLOSET_DIR)

copy_pairs(final_train, "train", YOLOSET_DIR)
copy_pairs(final_val, "val", YOLOSET_DIR)
if len(final_test) > 0:
    copy_pairs(final_test, "test", YOLOSET_DIR)

print("\nSplit done:")
print(" train:", len(final_train), f"(pos={len(train_pos)}, neg={len(train_neg)})")
print(" val:  ", len(final_val), f"(pos={len(val_pos)}, neg={len(val_neg)})")
print(" test: ", len(final_test), f"(pos={len(test_pos)}, neg={len(test_neg)})")

if CREATE_DATA_YAML:
    yaml_path = YOLOSET_DIR / "data.yaml"
    yaml_text = "\n".join([
        f"path: {YOLOSET_DIR.resolve().as_posix()}",
        "train: train/images",
        "val: val/images",
        "test: test/images",
        f"nc: {len(CLASS_NAMES)}",
        "names: [" + ", ".join(CLASS_NAMES) + "]"
    ])
    yaml_path.write_text(yaml_text, encoding="utf-8")
    print("Created:", yaml_path)

print("\nReady for YOLO training from:", YOLOSET_DIR)


All exported pairs: 0
Positive pairs: 0
Negative pairs: 0
Selected positives: 0
Selected negatives: 0

Split done:
 train: 0 (pos=0, neg=0)
 val:   0 (pos=0, neg=0)
 test:  0 (pos=0, neg=0)
Created: dataset_out_v6\yolo_split\data.yaml

Ready for YOLO training from: dataset_out_v6\yolo_split
